# PRAKTIK 3: DEMONSTRASI PROXIMAL SOIL SENSING (PSS)
## Analisis Kemometrika Spektroskopi Pantulan Tanah Vis-NIR (350–2500 nm) untuk C-Organik
**Pelatihan Pemanfaatan RS dan DSM untuk Pertanian Presisi**  
Pusat Studi Pengelolaan Sumber Daya Lahan (PS2DL) UGM & PT SMART Tbk

---
> **Petunjuk Lingkungan Kerja (Virtual Environment / venv):**  
> Pastikan notebook ini dijalankan pada kernel virtual environment **`Python (Pelatihan SMART)`** (`env_smart`).  
> Jika belum membuat venv, jalankan skrip penyiapan otomatis:  
> - **Linux / macOS:** `./setup_env.sh` (pada folder `Output/Pelatihan/`)  
> - **Windows:** `setup_env.bat` (pada folder `Output\Pelatihan\`)  
> Periksa pojok kanan atas jendela JupyterLab; pastikan kernel yang aktif tertulis `Python (Pelatihan SMART)`.

---
### Tujuan Praktikum:
1. Membaca dan memvisualisasikan spektra tanah mentah (*raw reflectance spectra*) pada rentang Vis-NIR (350–2500 nm).
2. Menerapkan transformasi praolah sinyal kemometrika: Absorbansi ($A = \log(1/R)$), *Standard Normal Variate* (SNV), dan turunan pertama Savitzky–Golay.
3. Mengidentifikasi puncak serapan fisis diagnostik bahan organik tanah (SOM 450–650 nm, ikatan C-H 1720 & 2300 nm) serta gugus hidroksil (O-H 1400 & 1900 nm).
4. Melatih dan memvalidasi model *Partial Least Squares Regression* (PLSR) untuk estimasi kuantitatif kadar C-Organik tanah.

In [ ]:
# 1. Inisialisasi Pustaka Python
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

print('Pustaka berhasil diimpor dengan sukses!')

### 2. Memuat Dataset Spektroskopi Tanah Vis-NIR (852 Sampel)
Kita membaca berkas `spektra_visnir_857sampel.csv` yang berisi spektra kontinu 350–2500 nm (resolusi 1 nm) berpasangan dengan hasil uji laboratorium kadar C-Organik (%).

In [ ]:
# 2. Membaca data spektroskopi CSV
csv_path = '../data/praktik-3/spektra_visnir_857sampel.csv'
if not os.path.exists(csv_path):
    csv_path = 'Output/Pelatihan/data/praktik-3/spektra_visnir_857sampel.csv'

with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    corg_idx = header.index('C_Organik')
    wavelengths = np.array([int(w) for w in header[9:]])
    
    X_list = []
    y_list = []
    for row in reader:
        try:
            val_c = float(row[corg_idx])
            if val_c > 0:
                X_list.append([float(v) for v in row[9:]])
                y_list.append(val_c)
        except:
            pass

spectra_raw = np.array(X_list)
c_org = np.array(y_list)

print(f'Total sampel tanah : {len(c_org)} sampel')
print(f'Jumlah pita spektral: {len(wavelengths)} panjang gelombang ({wavelengths[0]} - {wavelengths[-1]} nm)')
print(f'Rentang C-Organik    : {c_org.min():.2f}% - {c_org.max():.2f}% (Rata-rata: {c_org.mean():.2f}%)')

### 3. Visualisasi Kurva Spektral Mentah (*Raw Reflectance*)
Perhatikan perbedaan kurva pantulan: Sampel dengan C-Organik tinggi memiliki reflektansi gelap (kurva di bawah 0.25) dan lereng curam di rentang tampak (400–700 nm) akibat penyerapan kromofor humus.

In [ ]:
# 3. Plot Kurva Spektral Mentah
plt.figure(figsize=(10, 5), dpi=150)
idx_high_corg = np.where(c_org > 2.0)[0]
idx_low_corg = np.where(c_org < 0.8)[0]

for idx in idx_low_corg[:5]:
    plt.plot(wavelengths, spectra_raw[idx], color='#E76F51', alpha=0.7, label='C-Organik Rendah (<0.8%)' if idx == idx_low_corg[0] else '')
for idx in idx_high_corg[:5]:
    plt.plot(wavelengths, spectra_raw[idx], color='#264653', alpha=0.9, linewidth=1.5, label='C-Organik Tinggi (>2.0%)' if idx == idx_high_corg[0] else '')

plt.title('Kurva Spektrum Reflektansi Mentah Tanah (Vis-NIR 350 - 2500 nm)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Panjang Gelombang (nm)', fontsize=11)
plt.ylabel('Faktor Reflektansi (Reflectance)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 4. Praolah Sinyal Kemometrika (Absorbansi, SNV, & Savitzky–Golay)
Sinyal mentah mengandung derau akibat hamburan ukuran butir tanah. Kita terapkan 3 tahap praolah:
1. **Absorbansi:** $A = \log_{10}(1 / R)$
2. **Standard Normal Variate (SNV):** Menghilangkan variasi lereng dan offset baseline antar-sampel.
3. **Savitzky–Golay First Derivative:** Mempertegas titik belok dan puncak serapan diagnostik senyawa organik.

In [ ]:
# 4. Transformasi Absorbansi, SNV, dan Savitzky-Golay
spectra_safe = np.clip(spectra_raw, 1e-4, 1.0)
absorbance = np.log10(1.0 / spectra_safe)

mean_abs = np.mean(absorbance, axis=1, keepdims=True)
std_abs = np.std(absorbance, axis=1, keepdims=True)
spectra_snv = (absorbance - mean_abs) / std_abs

# Savitzky-Golay Filter (Window=15, Polyorder=2, Deriv=1)
spectra_sg_deriv1 = savgol_filter(spectra_snv, window_length=15, polyorder=2, deriv=1, axis=1)

print('Praolah sinyal SNV dan Savitzky-Golay turunan pertama berhasil dihitung!')

### 5. Visualisasi Fitur Serapan Diagnostik C-Organik
Perhatikan bagaimana turunan pertama SNV menonjolkan fitur serapan kromofor bahan organik tanah di **450–650 nm**, ikatan C-H alifatik di **1720 nm**, ikatan C-H + C=O di **2300 nm**, serta serapan air di **1400 nm** dan **1900 nm**.

In [ ]:
# 5. Plot Perbandingan Hasil Praolah
fig, axes = plt.subplots(2, 1, figsize=(11, 8), dpi=150, sharex=True)

# Panel atas: Spektra ter-SNV
for idx in idx_high_corg[:4]:
    axes[0].plot(wavelengths, spectra_snv[idx], color='#01416B', alpha=0.8)
axes[0].set_title('Spektra Absorbansi Ternormalisasi SNV', fontsize=12, fontweight='bold')
axes[0].set_ylabel('SNV Absorbansi', fontsize=10)
axes[0].axvline(550, color='#8c510a', linestyle='--', label='Kromofor SOM (450-650 nm)')
axes[0].axvline(1400, color='gray', linestyle=':', label='O-H (1400 nm)')
axes[0].axvline(1720, color='#d95f02', linestyle='-.', label='C-H Alifatik (1720 nm)')
axes[0].axvline(1900, color='blue', linestyle=':', label='H2O (1900 nm)')
axes[0].axvline(2300, color='#7570b3', linestyle='-.', label='C-H + C=O Humus (2300 nm)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)

# Panel bawah: Turunan Pertama Savitzky-Golay
for idx in idx_high_corg[:4]:
    axes[1].plot(wavelengths, spectra_sg_deriv1[idx], color='#2A9D8F', alpha=0.8)
axes[1].set_title('Turunan Pertama Savitzky–Golay (Puncak Serapan Diagnostik C-Organik)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Panjang Gelombang (nm)', fontsize=11)
axes[1].set_ylabel('Turunan Ke-1', fontsize=10)
axes[1].axvline(550, color='#8c510a', linestyle='--')
axes[1].axvline(1400, color='gray', linestyle=':')
axes[1].axvline(1720, color='#d95f02', linestyle='-.')
axes[1].axvline(1900, color='blue', linestyle=':')
axes[1].axvline(2300, color='#7570b3', linestyle='-.')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 6. Pembangunan Model Kalibrasi Spektral (PLSR)
Kita melatih algoritma *Partial Least Squares Regression* (PLSR) untuk mengestimasi kadar C-Organik (%) dari spektrum praolah.

In [ ]:
# 6. Pelatihan dan Evaluasi Model PLSR C-Organik
X = spectra_sg_deriv1
y = c_org

# Pembagian 75% data latih dan 25% data validasi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Kalibrasi PLSR dengan 6 komponen laten
pls_model = PLSRegression(n_components=6)
pls_model.fit(X_train, y_train)

y_pred_train = pls_model.predict(X_train).ravel()
y_pred_test = pls_model.predict(X_test).ravel()

r2_train = r2_score(y_train, y_pred_train)
rmse_train = root_mean_squared_error(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = root_mean_squared_error(y_test, y_pred_test)
rpd_test = np.std(y_test) / rmse_test

print('=== HASIL EVALUASI MODEL PLSR PREDIKSI C-ORGANIK ===')
print(f'Kalibrasi Train R² : {r2_train:.3f} | RMSE: {rmse_train:.3f}%')
print(f'Validasi Test R²   : {r2_test:.3f} | RMSE: {rmse_test:.3f}%')
print(f'Rasio Kinerja RPD  : {rpd_test:.2f}')

### 7. Scatterplot Kalibrasi: Nilai Lab Aktual vs Prediksi C-Organik PSS
Grafik sebaran 1:1 antara hasil uji laboratorium basah vs nilai estimasi spektrometer Vis-NIR.

In [ ]:
# 7. Plot Korelasi 1:1 Uji Lab vs Prediksi Spektroskopi C-Organik
plt.figure(figsize=(6.5, 6.5), dpi=150)
plt.scatter(y_train, y_pred_train, color='#1f78b4', alpha=0.5, s=35, label=f'Data Kalibrasi (R² = {r2_train:.2f})')
plt.scatter(y_test, y_pred_test, color='#e31a1c', edgecolors='k', linewidth=0.5, s=45, alpha=0.8, label=f'Data Validasi (R² = {r2_test:.2f}, RPD = {rpd_test:.2f})')

lims = [0, max(y.max(), y_pred_test.max()) + 0.5]
plt.plot(lims, lims, color='black', linestyle='--', linewidth=1.5, label='Garis 1:1 Ideal')
plt.xlim(lims)
plt.ylim(lims)

plt.title(f'Akurasi Estimasi PSS Vis-NIR: Kadar C-Organik (%)\nRMSE Validasi: {rmse_test:.2f}% | RPD: {rpd_test:.2f}', fontsize=11, fontweight='bold', pad=12)
plt.xlabel('Nilai Uji Laboratorium Aktual (%)', fontsize=11)
plt.ylabel('Nilai Prediksi Spektroskopi Vis-NIR (%)', fontsize=11)
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Kesimpulan Praktik 3:
Spektroskopi reflektansi Vis-NIR terbukti mampu mendeteksi gugus kromofor bahan organik (SOM) dan ikatan C-H alifatik tanah, memungkinkan estimasi cepat C-Organik tanah secara non-destruktif dan real-time di perkebunan.